In [33]:
import os
import glob
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_score, recall_score, f1_score
import warnings
warnings.filterwarnings('ignore')

In [34]:
BASE_PATH = "/kaggle/input/datasets/revsyko/insider-threat-cert-4-2/r4.2/"

In [35]:
def basic_eda(df, dataset_name):
    print(f"\n{'='*50}")
    print(f" Exploratory Data Analysis: {dataset_name.upper()} ")
    print(f"{'='*50}")
    print(f"Shape: {df.shape[0]:,} rows | {df.shape[1]} columns\n")
 
    summary_df = pd.DataFrame({
        'Data Type': df.dtypes,
        'Total Values': df.count(),
        'Null Values': df.isnull().sum(),
        'Unique Values': df.nunique()
    })
    print(summary_df.to_string())
    print("\nData Preview (First 3 Rows):")
    try:
        display(df.head(3))  # Jupyter/Kaggle
    except NameError:
        print(df.head(3))
    print("\n")

In [36]:
def _logon_features(logon):
    print("Processing Logon features...")
    logon["day"] = logon["date"].dt.date
    logon["hour"] = logon["date"].dt.hour
    logon["is_off_hours"] = ((logon["hour"] < 7) | (logon["hour"] >= 18)).astype(int)
    return logon.groupby(["user", "day"]).agg(
        logon_count=("activity", lambda x: (x == "Logon").sum()),
        off_hours_logons=("is_off_hours", "sum"),
        distinct_pcs=("pc", "nunique")
    ).reset_index()
 
 
def _device_features(device):
    print("Processing Device features...")
    device["day"] = device["date"].dt.date
    device["hour"] = device["date"].dt.hour
    connects = device[device["activity"] == "Connect"].copy()
    connects["is_off_hours"] = ((connects["hour"] < 7) | (connects["hour"] >= 18)).astype(int)
    return connects.groupby(["user", "day"]).agg(
        usb_connects=("activity", "count"),
        off_hours_usb=("is_off_hours", "sum")
    ).reset_index()
 
 
def _file_features(file_df):
    print("Processing File features...")
    file_df["day"] = file_df["date"].dt.date
    file_df["is_sensitive"] = file_df["filename"].str.contains(
        r'\.doc|\.pdf|\.zip', case=False, na=False
    ).astype(int)
    return file_df.groupby(["user", "day"]).agg(
        files_copied_to_usb=("filename", "count"),
        sensitive_files_to_usb=("is_sensitive", "sum")
    ).reset_index()
 
 
def _email_features(email):
    print("Processing Email features...")
    email["day"] = email["date"].dt.date
    email["is_external"] = (~email["to"].str.contains("dtaa.com", case=False, na=False)).astype(int)
 
    if "attachment_count" in email.columns:
        email["attachment_count"] = pd.to_numeric(email["attachment_count"], errors="coerce").fillna(0)
    elif "attachments" in email.columns:
        email["attachment_count"] = pd.to_numeric(email["attachments"], errors="coerce").fillna(0)
    else:
        email["attachment_count"] = 0
 
    return email.groupby(["user", "day"]).agg(
        total_emails_sent=("date", "count"),
        external_emails_sent=("is_external", "sum"),
        total_attachments=("attachment_count", "sum"),
        total_email_size=("size", "sum")
    ).reset_index()

In [37]:
def _http_features_and_ids_chunked(path, chunksize=2_000_000):
    print("Processing HTTP features (chunked, full file)...")
    agg_parts = []
    id_parts = []
    total_rows = 0
    for chunk in pd.read_csv(path, parse_dates=["date"], chunksize=chunksize):
        total_rows += len(chunk)
        chunk["day"] = chunk["date"].dt.date
        chunk["is_cloud_or_job"] = chunk["url"].str.contains(
            "dropbox|drive|monster|linkedin", case=False, na=False
        ).astype(int)

        part = chunk.groupby(["user", "day"]).agg(
            http_requests=("date", "count"),
            cloud_job_visits=("is_cloud_or_job", "sum")
        ).reset_index()
        agg_parts.append(part)

        id_parts.append(chunk[["id", "user", "day"]])

        print(f"  ...processed {total_rows:,} http rows so far")

    full = pd.concat(agg_parts, ignore_index=True)
    # A given user/day pair can be split across chunk boundaries -> re-sum
    features = full.groupby(["user", "day"]).sum().reset_index()

    ids = pd.concat(id_parts, ignore_index=True)
    return features, ids

In [ ]:
logon = pd.read_csv(os.path.join(BASE_PATH, "logon.csv"), parse_dates=["date"])
device = pd.read_csv(os.path.join(BASE_PATH, "device.csv"), parse_dates=["date"])
file_df = pd.read_csv(os.path.join(BASE_PATH, "file.csv"), parse_dates=["date"])
email = pd.read_csv(os.path.join(BASE_PATH, "email.csv"), parse_dates=["date"])
 
datasets_to_inspect = {
    "Logon": logon,
    "Device": device,
    "File": file_df,
    "Email": email,
}
for name, df in datasets_to_inspect.items():
    basic_eda(df, name)

In [ ]:
print("\nHTTP file will be processed in chunks (not loaded fully into memory for EDA).")
http_preview = pd.read_csv(os.path.join(BASE_PATH, "http.csv"), nrows=5)
print("HTTP columns:", list(http_preview.columns))

In [ ]:
print("\nCapturing id/user/day tables for later label matching...")
id_tables = {}
for name, df in [("logon", logon), ("device", device), ("file", file_df), ("email", email)]:
    # "day" gets added inside the feature functions below via df["day"] = ...date,
    # but we need it here too, so compute it once, up front, consistently.
    df["day"] = df["date"].dt.date
    id_tables[name] = df[["id", "user", "day"]].copy()
    print(f"  {name}: {len(id_tables[name]):,} rows captured")

In [ ]:
print("\nInitiating Feature Extraction Pipeline...")
combined_df = _logon_features(logon)
 
for feat_func, df in [(_device_features, device), (_file_features, file_df), (_email_features, email)]:
    combined_df = combined_df.merge(feat_func(df), on=["user", "day"], how="outer")

In [ ]:
http_feat, http_ids = _http_features_and_ids_chunked(os.path.join(BASE_PATH, "http.csv"))
combined_df = combined_df.merge(http_feat, on=["user", "day"], how="outer")
id_tables["http"] = http_ids
print(f"  http: {len(id_tables['http']):,} rows captured")

numeric_cols = combined_df.columns.difference(["user", "day"])
combined_df[numeric_cols] = combined_df[numeric_cols].fillna(0)

In [ ]:
os.makedirs("/kaggle/working", exist_ok=True)
combined_df.to_csv("/kaggle/working/combined_df_pre_ldap.csv", index=False)
print(f"Saved combined_df_pre_ldap.csv — {combined_df.shape[0]:,} rows, {combined_df.shape[1]} columns")

In [38]:
pre_ldap = "/kaggle/input/datasets/revsyko/combined-df-pre-ldap/combined_df_pre_ldap.csv"

import pandas as pd

combined_df = pd.read_csv(pre_ldap)

In [40]:

numeric_cols = combined_df.columns.difference(["user", "day"])
combined_df[numeric_cols] = combined_df[numeric_cols].fillna(0)

In [41]:
basic_eda(combined_df, "Combined Features (pre-LDAP)")
print("Combined (pre-LDAP) numeric summary:")
print(combined_df[numeric_cols].describe().T.to_string())
print(f"\nDistinct users: {combined_df['user'].nunique():,} | Distinct user-days: {len(combined_df):,}")
print(f"Date range: {combined_df['day'].min()} to {combined_df['day'].max()}")


 Exploratory Data Analysis: COMBINED FEATURES (PRE-LDAP) 
Shape: 330,452 rows | 15 columns

                       Data Type  Total Values  Null Values  Unique Values
user                      object        330452            0           1000
day                       object        330452            0            501
logon_count                int64        330452            0             10
off_hours_logons           int64        330452            0             14
distinct_pcs               int64        330452            0              7
usb_connects             float64        330452            0             15
off_hours_usb            float64        330452            0             11
files_copied_to_usb      float64        330452            0             54
sensitive_files_to_usb   float64        330452            0             49
total_emails_sent        float64        330452            0             37
external_emails_sent     float64        330452            0             34
total_a

,user,day,logon_count,off_hours_logons,distinct_pcs,usb_connects,off_hours_usb,files_copied_to_usb,sensitive_files_to_usb,total_emails_sent,external_emails_sent,total_attachments,total_email_size,http_requests,cloud_job_visits
0,AAE0190,2010-01-04,1,1,1,0.0,0.0,0.0,0.0,14.0,1.0,4.0,441328.0,143.0,1.0
1,AAE0190,2010-01-05,1,1,1,0.0,0.0,0.0,0.0,13.0,5.0,2.0,355552.0,143.0,3.0
2,AAE0190,2010-01-06,1,1,1,0.0,0.0,0.0,0.0,14.0,6.0,12.0,532647.0,143.0,7.0




Combined (pre-LDAP) numeric summary:
                           count           mean            std  min       25%       50%        75%        max
cloud_job_visits        330452.0       1.724686       3.568720  0.0      0.00       0.0       2.00       68.0
distinct_pcs            330452.0       1.151943       0.642662  1.0      1.00       1.0       1.00        7.0
external_emails_sent    330452.0       3.160265       3.443971  0.0      1.00       2.0       5.00       33.0
files_copied_to_usb     330452.0       1.348399       4.697557  0.0      0.00       0.0       0.00       54.0
http_requests           330452.0      86.047060      57.281695  0.0     29.00      95.0     114.00      400.0
logon_count             330452.0       1.424083       0.795866  0.0      1.00       1.0       2.00        9.0
off_hours_logons        330452.0       0.587392       1.095589  0.0      0.00       0.0       1.00       13.0
off_hours_usb           330452.0       0.036293       0.316298  0.0      0.00    

In [42]:
# ---------------------------------------------------------
# 6. LDAP Integration
# ---------------------------------------------------------
print("\nProcessing Monthly LDAP records...")
ldap_path = os.path.join(BASE_PATH, "LDAP")
all_ldap_files = glob.glob(os.path.join(ldap_path, "*.csv"))

ldap_list = []
for file in all_ldap_files:
    df = pd.read_csv(file)
    month_str = os.path.basename(file).split('.csv')[0]
    df['month_year'] = month_str
    ldap_list.append(df)

master_ldap = pd.concat(ldap_list, ignore_index=True)
master_ldap = master_ldap[['user_id', 'month_year', 'role', 'department', 'team', 'supervisor']]
master_ldap.rename(columns={'user_id': 'user'}, inplace=True)

# FIX: dedupe defensively before merging — if any monthly LDAP file has more
# than one row for the same (user, month), a plain merge fans out silently
# and duplicates every activity row for that user that month. Guard against it.
before_dedupe = len(master_ldap)
master_ldap = master_ldap.drop_duplicates(subset=['user', 'month_year'], keep='last')
if len(master_ldap) != before_dedupe:
    print(f"  Dropped {before_dedupe - len(master_ldap)} duplicate LDAP (user, month) entries")

combined_df['day'] = pd.to_datetime(combined_df['day'])
combined_df['month_year'] = combined_df['day'].dt.strftime('%Y-%m')

rows_before_ldap_merge = len(combined_df)
combined_df = pd.merge(combined_df, master_ldap, on=['user', 'month_year'], how='left')
assert len(combined_df) == rows_before_ldap_merge, (
    f"LDAP merge caused row duplication: {rows_before_ldap_merge} -> {len(combined_df)}. "
    f"Check master_ldap for remaining duplicate (user, month_year) pairs."
)
print(f"  LDAP merge OK — row count unchanged ({rows_before_ldap_merge:,} rows)")

combined_df = combined_df.sort_values(by=['user', 'day'])
cat_columns = ['role', 'department', 'team', 'supervisor']
combined_df[cat_columns] = combined_df.groupby('user')[cat_columns].ffill()

le_dict = {}
for col in cat_columns:
    combined_df[col] = combined_df[col].fillna('Unknown').astype(str)
    le = LabelEncoder()
    combined_df[f"{col}_encoded"] = le.fit_transform(combined_df[col])
    le_dict[col] = le

combined_df.drop(columns=['role', 'department', 'team', 'supervisor', 'month_year'], inplace=True)


Processing Monthly LDAP records...
  LDAP merge OK — row count unchanged (330,452 rows)


In [43]:
basic_eda(combined_df, "Combined Features (final, post-LDAP)")
print("Final numeric summary:")
final_numeric_cols = combined_df.columns.difference(["user", "day"])
print(combined_df[final_numeric_cols].describe().T.to_string())


 Exploratory Data Analysis: COMBINED FEATURES (FINAL, POST-LDAP) 
Shape: 330,452 rows | 19 columns

                             Data Type  Total Values  Null Values  Unique Values
user                            object        330452            0           1000
day                     datetime64[ns]        330452            0            501
logon_count                      int64        330452            0             10
off_hours_logons                 int64        330452            0             14
distinct_pcs                     int64        330452            0              7
usb_connects                   float64        330452            0             15
off_hours_usb                  float64        330452            0             11
files_copied_to_usb            float64        330452            0             54
sensitive_files_to_usb         float64        330452            0             49
total_emails_sent              float64        330452            0             37
external

,user,day,logon_count,off_hours_logons,distinct_pcs,usb_connects,off_hours_usb,files_copied_to_usb,sensitive_files_to_usb,total_emails_sent,external_emails_sent,total_attachments,total_email_size,http_requests,cloud_job_visits,role_encoded,department_encoded,team_encoded,supervisor_encoded
0,AAE0190,2010-01-04,1,1,1,0.0,0.0,0.0,0.0,14.0,1.0,4.0,441328.0,143.0,1.0,22,4,38,43
1,AAE0190,2010-01-05,1,1,1,0.0,0.0,0.0,0.0,13.0,5.0,2.0,355552.0,143.0,3.0,22,4,38,43
2,AAE0190,2010-01-06,1,1,1,0.0,0.0,0.0,0.0,14.0,6.0,12.0,532647.0,143.0,7.0,22,4,38,43




Final numeric summary:
                           count           mean            std  min       25%       50%        75%        max
cloud_job_visits        330452.0       1.724686       3.568720  0.0      0.00       0.0       2.00       68.0
department_encoded      330452.0      11.970020       4.323835  0.0     10.00      12.0      15.00       22.0
distinct_pcs            330452.0       1.151943       0.642662  1.0      1.00       1.0       1.00        7.0
external_emails_sent    330452.0       3.160265       3.443971  0.0      1.00       2.0       5.00       33.0
files_copied_to_usb     330452.0       1.348399       4.697557  0.0      0.00       0.0       0.00       54.0
http_requests           330452.0      86.047060      57.281695  0.0     29.00      95.0     114.00      400.0
logon_count             330452.0       1.424083       0.795866  0.0      1.00       1.0       2.00        9.0
off_hours_logons        330452.0       0.587392       1.095589  0.0      0.00       0.0       1

In [44]:
encoded_cols = [c for c in combined_df.columns if c.endswith("_encoded")]
print("\nLDAP merge coverage check (rows where LDAP context could not be found/ffilled):")
for col in encoded_cols:
    base_col = col.replace("_encoded", "")
    unknown_label = le_dict[base_col].transform(['Unknown'])[0] if 'Unknown' in le_dict[base_col].classes_ else None
    if unknown_label is not None:
        n_unknown = (combined_df[col] == unknown_label).sum()
        pct_unknown = 100 * n_unknown / len(combined_df)
        print(f"  {base_col}: {n_unknown:,} rows ({pct_unknown:.2f}%) fell back to 'Unknown'")

print(f"\nPipeline Complete. Final Matrix Shape: {combined_df.shape}")
display(combined_df.head(3))


LDAP merge coverage check (rows where LDAP context could not be found/ffilled):
  department: 4,844 rows (1.47%) fell back to 'Unknown'
  team: 43,304 rows (13.10%) fell back to 'Unknown'
  supervisor: 346 rows (0.10%) fell back to 'Unknown'

Pipeline Complete. Final Matrix Shape: (330452, 19)


,user,day,logon_count,off_hours_logons,distinct_pcs,usb_connects,off_hours_usb,files_copied_to_usb,sensitive_files_to_usb,total_emails_sent,external_emails_sent,total_attachments,total_email_size,http_requests,cloud_job_visits,role_encoded,department_encoded,team_encoded,supervisor_encoded
0,AAE0190,2010-01-04,1,1,1,0.0,0.0,0.0,0.0,14.0,1.0,4.0,441328.0,143.0,1.0,22,4,38,43
1,AAE0190,2010-01-05,1,1,1,0.0,0.0,0.0,0.0,13.0,5.0,2.0,355552.0,143.0,3.0,22,4,38,43
2,AAE0190,2010-01-06,1,1,1,0.0,0.0,0.0,0.0,14.0,6.0,12.0,532647.0,143.0,7.0,22,4,38,43


In [45]:
# Check: does the user have ANY LDAP record at all (any month), just missing team specifically?
unknown_team_users = combined_df[combined_df['team_encoded'] == 
    le_dict['team'].transform(['Unknown'])[0]]['user'].unique()

users_with_some_ldap = set(master_ldap['user'].unique())
truly_never_in_ldap = set(unknown_team_users) - users_with_some_ldap

print(f"Users with 'Unknown' team: {len(unknown_team_users)}")
print(f"...of those, never appear in LDAP at all: {len(truly_never_in_ldap)}")
print(f"...of those, appear in LDAP but team field itself was blank: {len(unknown_team_users) - len(truly_never_in_ldap)}")

Users with 'Unknown' team: 127
...of those, never appear in LDAP at all: 0
...of those, appear in LDAP but team field itself was blank: 127


In [46]:

try:
    print(id_tables.keys())
    print({k: len(v) for k, v in id_tables.items()})
except NameError:
    print("id_tables is NOT in memory — needs rebuilding")

dict_keys(['logon', 'device', 'file', 'email', 'http'])
{'logon': 854859, 'device': 405380, 'file': 445581, 'email': 2629979, 'http': 28434423}


In [47]:
import pickle  # add this if starting fresh

def _http_ids_only_chunked(path, chunksize=2_000_000):
    parts = []
    total = 0
    for chunk in pd.read_csv(path, usecols=["id","date","user"], parse_dates=["date"], chunksize=chunksize):
        total += len(chunk)
        chunk["day"] = chunk["date"].dt.date
        parts.append(chunk[["id","user","day"]])
        print(f"  ...{total:,} rows")
    return pd.concat(parts, ignore_index=True)

logon = pd.read_csv(os.path.join(BASE_PATH,"logon.csv"), parse_dates=["date"])
device = pd.read_csv(os.path.join(BASE_PATH,"device.csv"), parse_dates=["date"])
file_df = pd.read_csv(os.path.join(BASE_PATH,"file.csv"), parse_dates=["date"])
email = pd.read_csv(os.path.join(BASE_PATH,"email.csv"), parse_dates=["date"])
for df in [logon, device, file_df, email]:
    df["day"] = df["date"].dt.date

id_tables = {
    "logon": logon[["id","user","day"]].copy(),
    "device": device[["id","user","day"]].copy(),
    "file": file_df[["id","user","day"]].copy(),
    "email": email[["id","user","day"]].copy(),
    "http": _http_ids_only_chunked(os.path.join(BASE_PATH,"http.csv")),
}

for k, v in id_tables.items():
    print(f"{k}: {len(v):,} rows")

with open("/kaggle/working/id_tables.pkl", "wb") as f:
    pickle.dump(id_tables, f)
print("\nSaved id_tables.pkl")

  ...2,000,000 rows
  ...4,000,000 rows
  ...6,000,000 rows
  ...8,000,000 rows
  ...10,000,000 rows
  ...12,000,000 rows
  ...14,000,000 rows
  ...16,000,000 rows
  ...18,000,000 rows
  ...20,000,000 rows
  ...22,000,000 rows
  ...24,000,000 rows
  ...26,000,000 rows
  ...28,000,000 rows
  ...28,434,423 rows
logon: 854,859 rows
device: 405,380 rows
file: 445,581 rows
email: 2,629,979 rows
http: 28,434,423 rows

Saved id_tables.pkl


In [48]:
combined_df = combined_df.drop(columns=['team_encoded'])
print(f"Dropped team_encoded. Shape: {combined_df.shape}")

Dropped team_encoded. Shape: (330452, 18)


In [49]:
combined_df.to_csv("/kaggle/working/combined_df_final.csv", index=False)
print(f"Saved combined_df_final.csv — {combined_df.shape[0]:,} rows, {combined_df.shape[1]} columns")

Saved combined_df_final.csv — 330,452 rows, 18 columns


In [50]:
import glob

ANSWERS_BASE = "/kaggle/input/datasets/revsyko/answers/answers"  # adjust to your upload path

SCHEMAS = {
    "logon":  ["type", "id", "date", "user", "pc", "activity"],
    "device": ["type", "id", "date", "user", "pc", "activity"],
    "http":   ["type", "id", "date", "user", "pc", "url", "content"],
    "file":   ["type", "id", "date", "user", "pc", "filename", "content"],
    "email":  ["type", "id", "date", "user", "pc", "to", "cc", "bcc",
               "from", "size", "attachment_count", "content"],
}

def parse_answer_file(path):
    rows_by_type = {k: [] for k in SCHEMAS}
    with open(path, encoding="utf-8", errors="replace") as f:
        for line in f:
            line = line.rstrip("\r\n")
            if not line:
                continue
            row_type = line.split(",", 1)[0]
            if row_type not in SCHEMAS:
                continue
            n_fields = len(SCHEMAS[row_type])
            parts = line.split(",", n_fields - 1)
            if len(parts) == n_fields:
                rows_by_type[row_type].append(parts)
    return rows_by_type

def build_malicious_id_tables(answers_base, scenario_folders=("r4.2-1", "r4.2-2", "r4.2-3")):
    all_rows = {k: [] for k in SCHEMAS}
    for folder in scenario_folders:
        folder_path = os.path.join(answers_base, folder)
        for path in glob.glob(os.path.join(folder_path, "*.csv")):
            parsed = parse_answer_file(path)
            for k, rows in parsed.items():
                for r in rows:
                    all_rows[k].append(r + [folder])

    malicious_tables = {}
    for k in SCHEMAS:
        cols = SCHEMAS[k] + ["scenario"]
        df = pd.DataFrame(all_rows[k], columns=cols)
        malicious_tables[k] = df
        print(f"{k}: {len(df)} malicious events parsed")
    print(f"\nTotal: {sum(len(v) for v in malicious_tables.values())}")  # should be 7323
    return malicious_tables

malicious_tables = build_malicious_id_tables(ANSWERS_BASE)

mal_logon  = set(malicious_tables["logon"]["id"])
mal_device = set(malicious_tables["device"]["id"])
mal_http   = set(malicious_tables["http"]["id"])
mal_file   = set(malicious_tables["file"]["id"])
mal_email  = set(malicious_tables["email"]["id"])

logon: 198 malicious events parsed
device: 2785 malicious events parsed
http: 3860 malicious events parsed
file: 10 malicious events parsed
email: 470 malicious events parsed

Total: 7323


In [51]:
insiders = pd.read_csv(os.path.join(ANSWERS_BASE, "insiders.csv"))
insiders["dataset"] = insiders["dataset"].astype(str).str.strip()
insiders_42 = insiders[insiders["dataset"] == "4.2"].copy()
insiders_42["start_dt"] = pd.to_datetime(insiders_42["start"], errors="coerce")
insiders_42["end_dt"]   = pd.to_datetime(insiders_42["end"], errors="coerce")

print(f"insiders.csv lists {insiders_42['user'].nunique()} unique r4.2 insiders")

insiders.csv lists 70 unique r4.2 insiders


In [52]:
mal_id_sets = {"logon": mal_logon, "device": mal_device, "http": mal_http,
               "file": mal_file, "email": mal_email}

day_label_parts = []
for source, mal_ids in mal_id_sets.items():
    src_table = id_tables[source]
    matched = src_table[src_table["id"].isin(mal_ids)][["user", "day"]]
    day_label_parts.append(matched)
    print(f"{source}: matched {len(matched)} malicious rows in raw logs")

day_labels = pd.concat(day_label_parts, ignore_index=True).drop_duplicates()
day_labels["is_insider"] = 1
print(f"\nTotal unique malicious user-days: {len(day_labels)}")
print(f"Unique insiders matched: {day_labels['user'].nunique()} / 70")

logon: matched 198 malicious rows in raw logs
device: matched 2785 malicious rows in raw logs
http: matched 3860 malicious rows in raw logs
file: matched 10 malicious rows in raw logs
email: matched 470 malicious rows in raw logs

Total unique malicious user-days: 986
Unique insiders matched: 72 / 70


In [53]:
matched_users = set(day_labels["user"].unique())
true_insiders = set(insiders_42["user"].unique())

extra_users = matched_users - true_insiders
missing_users = true_insiders - matched_users

print(f"Matched but NOT in insiders.csv: {extra_users}")
print(f"In insiders.csv but NOT matched: {missing_users}")

Matched but NOT in insiders.csv: {'FAW0032', 'FBA0348'}
In insiders.csv but NOT matched: set()


In [54]:
for k, df in malicious_tables.items():
    hits = df[df["user"].isin(["FAW0032", "FBA0348"])]
    if len(hits):
        print(f"--- {k} ---")
        print(hits)

--- logon ---
      type                        id                 date     user       pc  \
142  logon  {F2X9-N6CT67WV-0165DWKQ}  10/01/2010 19:40:01  FBA0348  PC-8486   
143  logon  {B1I2-X3TE24CB-4083EMHJ}  10/01/2010 19:48:03  FBA0348  PC-8486   
148  logon  {B6Z6-A7NW63LF-0484QAYF}  06/18/2010 17:41:14  FAW0032  PC-5866   
149  logon  {F0S7-H0FB84GU-2311WDFJ}  06/18/2010 17:50:42  FAW0032  PC-5866   
154  logon  {U1J6-J3NA26AD-4968TGCF}  04/29/2011 19:56:47  FBA0348  PC-8486   
155  logon  {S1B9-Q6BH62ZK-9232AKPI}  04/29/2011 20:04:27  FBA0348  PC-8486   
160  logon  {W2L4-R0OW55DL-8306FICD}  07/23/2010 17:46:25  FAW0032  PC-5866   
161  logon  {N2Q2-J1AB95VW-2948TVCX}  07/23/2010 18:01:20  FAW0032  PC-5866   
166  logon  {X1R2-B1ZD80OM-7171TMPL}  06/11/2010 17:28:25  FAW0032  PC-5866   
167  logon  {P3I7-H9YD13RQ-2131ZCUP}  06/11/2010 17:42:48  FAW0032  PC-5866   
172  logon  {B2D7-M6YM43QI-3576HZNN}  10/15/2010 19:48:33  FAW0032  PC-5866   
173  logon  {B1D4-V3GR16EU-1309MTJH}  

In [55]:
import glob
files = glob.glob(os.path.join(ANSWERS_BASE, "r4.2-3", "*.csv"))
print(f"Files physically present in r4.2-3: {len(files)}")
for f in sorted(files):
    print(" ", os.path.basename(f))

Files physically present in r4.2-3: 10
  r4.2-3-BBS0039.csv
  r4.2-3-BSS0369.csv
  r4.2-3-CCA0046.csv
  r4.2-3-CSC0217.csv
  r4.2-3-GTD0219.csv
  r4.2-3-JGT0221.csv
  r4.2-3-JLM0364.csv
  r4.2-3-JTM0223.csv
  r4.2-3-MPM0220.csv
  r4.2-3-MSO0222.csv


In [56]:
for f in sorted(glob.glob(os.path.join(ANSWERS_BASE, "r4.2-3", "*.csv"))):
    with open(f, encoding="utf-8", errors="replace") as fh:
        content = fh.read()
        if "FAW0032" in content or "FBA0348" in content:
            print(os.path.basename(f))

r4.2-3-BBS0039.csv
r4.2-3-BSS0369.csv
r4.2-3-CCA0046.csv
r4.2-3-CSC0217.csv
r4.2-3-GTD0219.csv
r4.2-3-JGT0221.csv
r4.2-3-JLM0364.csv
r4.2-3-JTM0223.csv
r4.2-3-MPM0220.csv
r4.2-3-MSO0222.csv


In [57]:
with open("/kaggle/input/datasets/revsyko/answers/answers/scenarios.txt") as f:
    content = f.read()
# print the section describing scenario 3 specifically
idx = content.find("3")
print(content[max(0,idx-50):idx+1500])

tes than their previous activity) to steal data.

3. System administrator becomes disgruntled. Downloads a keylogger and
uses a thumb drive to transfer it to his supervisor's machine. The
next day, he uses the collected keylogs to log in as his supervisor
and send out an alarming mass email, causing panic in the
organization. He leaves the organization immediately.

4. A user logs into another user's machine and searches for
interesting files, emailing to their home email. This behavior occurs
more and more frequently over a 3 month period.

5. A member of a group decimated by layoffs uploads documents to
Dropbox, planning to use them for personal gain.



In [58]:
victim_accounts = {"FAW0032", "FBA0348"}
day_labels["label_type"] = day_labels["user"].apply(
    lambda u: "hijacked_credentials" if u in victim_accounts else "insider"
)

combined_df["day"] = pd.to_datetime(combined_df["day"]).dt.date
combined_df = combined_df.merge(day_labels[["user","day","is_insider"]], on=["user","day"], how="left")
combined_df["is_insider"] = combined_df["is_insider"].fillna(0).astype(int)

print(combined_df["is_insider"].value_counts())
print(f"Unique flagged identities: {combined_df[combined_df['is_insider']==1]['user'].nunique()} / 72")

is_insider
0    329466
1       986
Name: count, dtype: int64
Unique flagged identities: 72 / 72


In [59]:
combined_df.to_csv("/kaggle/working/combined_df_labeled.csv", index=False)
print(f"Saved combined_df_labeled.csv — {combined_df.shape}")

Saved combined_df_labeled.csv — (330452, 19)


In [60]:
feature_cols_by_source = {
    "logon": ["logon_count","off_hours_logons","distinct_pcs"],
    "device": ["usb_connects","off_hours_usb"],
    "file": ["files_copied_to_usb","sensitive_files_to_usb"],
    "email": ["total_emails_sent","external_emails_sent","total_attachments","total_email_size"],
    "http": ["http_requests","cloud_job_visits"],
}
insider_days = combined_df[combined_df["is_insider"]==1]
for src, cols in feature_cols_by_source.items():
    cov = (insider_days[cols].sum(axis=1) > 0).mean()
    print(f"{src}: {cov*100:.1f}% of insider-days show signal")

logon: 100.0% of insider-days show signal
device: 91.0% of insider-days show signal
file: 64.9% of insider-days show signal
email: 97.8% of insider-days show signal
http: 99.9% of insider-days show signal


In [61]:
combined_df = combined_df.sort_values(by="day").reset_index(drop=True)
split_idx = int(len(combined_df) * 0.70)
train_df = combined_df.iloc[:split_idx].copy()
test_df  = combined_df.iloc[split_idx:].copy()

drop_cols = ["user","day","is_insider"]
X_cols = [c for c in combined_df.columns if c not in drop_cols]

X_train, y_train = train_df[X_cols], train_df["is_insider"]
X_test, y_test   = test_df[X_cols], test_df["is_insider"]
print(f"Train: {len(X_train)} rows, {y_train.sum()} positive")
print(f"Test:  {len(X_test)} rows, {y_test.sum()} positive")

Train: 231316 rows, 689 positive
Test:  99136 rows, 297 positive


In [62]:
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

In [63]:
import xgboost as xgb
neg, pos = (y_train==0).sum(), (y_train==1).sum()
print(f"neg={neg}, pos={pos}, ratio={neg/pos:.1f}")

model = xgb.XGBClassifier(
    n_estimators=150, max_depth=6, learning_rate=0.05,
    scale_pos_weight=neg/pos, random_state=42, eval_metric="aucpr"
)
model.fit(X_train_scaled, y_train)

neg=230627, pos=689, ratio=334.7


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='aucpr', feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.05, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=150,
              n_jobs=None, num_parallel_tree=None, ...)

In [64]:
from sklearn.metrics import precision_score, recall_score, f1_score, precision_recall_curve, auc

y_pred = model.predict(X_test_scaled)
y_prob = model.predict_proba(X_test_scaled)[:,1]

precision, recall, _ = precision_recall_curve(y_test, y_prob)
pr_auc = auc(recall, precision)

print(f"PR-AUC: {pr_auc:.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall: {recall_score(y_test, y_pred):.4f}")
print(f"F1: {f1_score(y_test, y_pred):.4f}")

flagged_users = set(test_df.loc[y_pred==1, "user"])
true_insiders_test = set(test_df.loc[y_test==1, "user"])
print(f"User-level recall: {len(flagged_users & true_insiders_test)}/{len(true_insiders_test)}")

PR-AUC: 0.6471
Precision: 0.3015
Recall: 0.8283
F1: 0.4420
User-level recall: 23/25


In [ ]:
 [code]

In [ ]:
print("Assigning heuristic threat labels...")
# Define baseline means for threshold labeling
feature_columns = ['files_copied_to_usb', 'external_emails_sent', 'off_hours_logons', 'distinct_pcs', 'sensitive_files_to_usb', 'cloud_job_visits', 'off_hours_usb']
feature_means = combined_df[feature_columns].mean()

def assign_label(row):
    if row['files_copied_to_usb'] > feature_means['files_copied_to_usb'] * 1.5 or row['external_emails_sent'] > feature_means['external_emails_sent'] * 1.5:
        return 0 # Data Exfiltration
    elif row['off_hours_logons'] > feature_means['off_hours_logons'] * 1.5 or row['distinct_pcs'] > feature_means['distinct_pcs'] * 1.5:
        return 1 # IT Sabotage
    elif row['sensitive_files_to_usb'] > feature_means['sensitive_files_to_usb'] * 1.5 or row['cloud_job_visits'] > feature_means['cloud_job_visits'] * 1.5:
        return 2 # Intellectual Property Theft
    elif row['off_hours_usb'] > feature_means['off_hours_usb'] * 1.5:
        return 4 # Unauthorized Access
    else:
        return 3 # Normal

combined_df['Label'] = combined_df.apply(assign_label, axis=1)
print("Class Distribution:\n", combined_df['Label'].value_counts())

# Prepare X and y
X = combined_df.drop(columns=['user', 'day', 'Label'])
y = combined_df['Label']

# Scale features
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.3, stratify=y, random_state=42)

In [ ]:
 [code]

In [ ]:
 [code]